# Bank Fraud Detection - Model Training
Dataset: Credit Card Fraud Detection (Kaggle - mlg-ulb/creditcardfraud)

284,807 transactions, 492 fraudes (0.17%) — features V1-V28 (PCA), Time, Amount, Class

## 1. Import & téléchargement du dataset

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve, average_precision_score
from imblearn.over_sampling import SMOTE

path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
print("Dataset path:", path)

df = pd.read_csv(f"{path}/creditcard.csv")
df.shape

ModuleNotFoundError: No module named 'kagglehub'

## 2. Exploration rapide

In [ ]:
df.info()
print("\nClasses:")
print(df['Class'].value_counts())
print(f"\nTaux de fraude: {df['Class'].mean()*100:.3f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.countplot(x='Class', data=df, ax=axes[0])
axes[0].set_title('Distribution des classes (0=normal, 1=fraude)')
sns.histplot(df['Amount'], bins=50, ax=axes[1])
axes[1].set_title('Distribution des montants')
plt.tight_layout()
plt.show()

## 3. Préprocessing

In [ ]:
scaler = StandardScaler()
df['Amount_scaled'] = scaler.fit_transform(df[['Amount']])
df['Time_scaled'] = scaler.fit_transform(df[['Time']])
df = df.drop(['Amount', 'Time'], axis=1)

X = df.drop('Class', axis=1)
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Avant SMOTE:", y_train.value_counts().to_dict())

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("Après SMOTE:", y_train_res.value_counts().to_dict())

## 4. Entraînement - Random Forest

In [ ]:
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    n_jobs=-1,
    random_state=42,
    class_weight='balanced'
)

model.fit(X_train_res, y_train_res)

## 5. Évaluation

In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=['Normal', 'Fraude']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")
print(f"Average Precision (PR-AUC): {average_precision_score(y_test, y_proba):.4f}")

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Fraude'], yticklabels=['Normal', 'Fraude'])
plt.title('Matrice de confusion')
plt.ylabel('Réel')
plt.xlabel('Prédit')
plt.show()

In [ ]:
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
plt.figure(figsize=(8, 8))
importances.head(15).plot(kind='barh')
plt.title('Top 15 features les plus importantes')
plt.gca().invert_yaxis()
plt.show()

## 6. Sauvegarde du modèle (pour le backend FastAPI)

In [ ]:
joblib.dump(model, 'fraud_model.joblib')
joblib.dump(scaler, 'scaler.joblib')
joblib.dump(list(X.columns), 'feature_columns.joblib')

print("Modèle sauvegardé: fraud_model.joblib, scaler.joblib, feature_columns.joblib")
print("À copier dans backend/app/models/ pour le charger dans FastAPI")